In [1]:
# Install required libraries
!pip install -q albumentations
!pip install -q segmentation-models-pytorch

import os
# Mute OpenCV warnings BEFORE importing cv2
os.environ["OPENCV_LOG_LEVEL"] = "FATAL" 

import glob
import numpy as np
import cv2
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import albumentations as A
from albumentations.pytorch import ToTensorV2
import segmentation_models_pytorch as smp

# Set random seed for reproducibility
np.random.seed(42)
torch.manual_seed(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.8/154.8 kB 2.6 MB/s eta 0:00:0000:01
Using device: cuda


In [2]:
# 1. Dynamically locate the dataset
image_search_paths = glob.glob('/kaggle/input/**/train/images/*', recursive=True)
train_dir = os.path.dirname(os.path.dirname(image_search_paths[0]))
IMAGE_DIR = os.path.join(train_dir, 'images')
MASK_DIR = os.path.join(train_dir, 'gt')
ext = os.path.splitext(image_search_paths[0])[1]

cities = ['austin', 'chicago', 'kitsap', 'tyrol-w', 'vienna']
train_imgs, train_masks = [], []
val_imgs, val_masks = [], []

for city in cities:
    for i in range(1, 31):
        img_path = os.path.join(IMAGE_DIR, f"{city}{i}{ext}")
        mask_path = os.path.join(MASK_DIR, f"{city}{i}{ext}")
        if os.path.exists(img_path) and os.path.exists(mask_path):
            if i <= 25:
                train_imgs.append(img_path)
                train_masks.append(mask_path)
            elif i <= 30:
                val_imgs.append(img_path)
                val_masks.append(mask_path)

# 2. Define Augmentations
train_transform = A.Compose([
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.5),
    A.RandomRotate90(p=0.5),
    A.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1, p=0.3),
    ToTensorV2()
])

val_transform = A.Compose([ToTensorV2()])

# 3. Define Dataset Class
class InriaPatchDataset(Dataset):
    def __init__(self, image_paths, mask_paths, patch_size=256, stride=256, transform=None):
        self.patch_size = patch_size
        self.stride = stride
        self.transform = transform
        self.samples = []
        for img_p, msk_p in zip(image_paths, mask_paths):
            for y in range(0, 5000 - patch_size + 1, stride):
                for x in range(0, 5000 - patch_size + 1, stride):
                    self.samples.append((img_p, msk_p, x, y))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        img_path, msk_path, x, y = self.samples[idx]
        img = cv2.cvtColor(cv2.imread(img_path), cv2.COLOR_BGR2RGB)
        patch_img = img[y:y+self.patch_size, x:x+self.patch_size]

        mask = cv2.imread(msk_path, cv2.IMREAD_GRAYSCALE)
        patch_mask = (mask[y:y+self.patch_size, x:x+self.patch_size] > 128).astype(np.float32)

        if self.transform:
            augmented = self.transform(image=patch_img, mask=patch_mask)
            patch_img = augmented['image'].float() / 255.0
            patch_mask = augmented['mask'].unsqueeze(0)
        return patch_img, patch_mask

# Using a larger subset for the actual model
train_dataset = InriaPatchDataset(train_imgs, train_masks, transform=train_transform)
val_dataset = InriaPatchDataset(val_imgs, val_masks, transform=val_transform)

train_loader = DataLoader(torch.utils.data.Subset(train_dataset, torch.randperm(len(train_dataset))[:4000]), batch_size=16, shuffle=True, num_workers=2)
val_loader = DataLoader(torch.utils.data.Subset(val_dataset, torch.randperm(len(val_dataset))[:800]), batch_size=16, shuffle=False, num_workers=2)

In [6]:
# Initialize U-Net model
model = smp.Unet(
    encoder_name="resnet34",
    encoder_weights="imagenet",
    in_channels=3,
    classes=1,
    activation=None
).to(device)

# Compound Loss: BCE + Dice
class BCEDiceLoss(nn.Module):
    def __init__(self):
        super().__init__()
        self.bce = nn.BCEWithLogitsLoss()
        self.dice = smp.losses.DiceLoss(mode='binary', from_logits=True)

    def forward(self, y_pred, y_true):
        return 0.5 * self.bce(y_pred, y_true) + 0.5 * self.dice(y_pred, y_true)

criterion = BCEDiceLoss()
optimizer = optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-4)

def compute_metrics(pred_mask, true_mask, threshold=0.5, eps=1e-7):
    pred = (pred_mask > threshold).float()
    true = (true_mask > threshold).float()
    intersection = (pred * true).sum()
    union = pred.sum() + true.sum() - intersection
    iou = (intersection + eps) / (union + eps)
    dice = (2.0 * intersection + eps) / (pred.sum() + true.sum() + eps)
    return iou.item(), dice.item()

In [7]:
num_epochs = 10 # Increase for final training runs
best_iou = 0.0

for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0
    for images, masks in train_loader:
        images, masks = images.to(device), masks.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, masks)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()
        
    # Validation
    model.eval()
    val_iou_list, val_dice_list = [], []
    with torch.no_grad():
        for images, masks in val_loader:
            images, masks = images.to(device), masks.to(device)
            preds = torch.sigmoid(model(images))
            iou, dice = compute_metrics(preds, masks)
            val_iou_list.append(iou)
            val_dice_list.append(dice)
            
    avg_iou = np.mean(val_iou_list)
    avg_dice = np.mean(val_dice_list)
    print(f"Epoch [{epoch+1}/{num_epochs}] | Loss: {running_loss/len(train_loader):.4f} | Val IoU: {avg_iou:.4f}")

    if avg_iou > best_iou:
        best_iou = avg_iou
        torch.save(model.state_dict(), '/kaggle/working/best_unet_resnet34.pth')
        print("--> Saved new best model checkpoint!")

Epoch [1/10] | Loss: 0.4251 | Val IoU: 0.5964
--> Saved new best model checkpoint!
Epoch [2/10] | Loss: 0.2698 | Val IoU: 0.6357
--> Saved new best model checkpoint!
Epoch [3/10] | Loss: 0.2249 | Val IoU: 0.6300
Epoch [4/10] | Loss: 0.2102 | Val IoU: 0.6341
Epoch [5/10] | Loss: 0.1983 | Val IoU: 0.6939
--> Saved new best model checkpoint!
Epoch [6/10] | Loss: 0.1866 | Val IoU: 0.6893
Epoch [7/10] | Loss: 0.1849 | Val IoU: 0.7001
--> Saved new best model checkpoint!
Epoch [8/10] | Loss: 0.1843 | Val IoU: 0.6892
Epoch [9/10] | Loss: 0.1675 | Val IoU: 0.7043
--> Saved new best model checkpoint!
Epoch [10/10] | Loss: 0.1651 | Val IoU: 0.7068
--> Saved new best model checkpoint!
